<center><h1>NY Liberty Analytics Project</h1></center>

In [1]:
import time
from nba_api.stats.endpoints import leaguegamelog
import pandas as pd

# 1. Set our variables (WNBA League ID is '10')
>    ## Define the exact 5 years we want

In [2]:
seasons = ['2021', '2022', '2023', '2024', '2025']
wnba_league_id = '10'

In [3]:
all_seasons_data = []

In [4]:
print("Connecting to API to fetch 5 years of WNBA data...")

Connecting to API to fetch 5 years of WNBA data...


# 2. Call the League Game Log API endpoint (Pulls all the teams!)

In [5]:
# 2. Loop through each year
for season in seasons:
    print(f"Fetching the {season} season...")
    
    # We use a try/except block just in case one year fails, it won't crash the whole script
    try:
        # Added timeout=60 to give the NBA server more time to respond!
        gamelog = leaguegamelog.LeagueGameLog(
            season=season, 
            league_id=wnba_league_id,
            timeout=60 
        )
        
        df_season = gamelog.get_data_frames()[0]
        all_seasons_data.append(df_season)
        
        # VERY IMPORTANT: Pause for 3 seconds before asking for the next year
        # This prevents the NBA server from blocking your IP address!
        time.sleep(3)
        
    except Exception as e:
        print(f"Uh oh, failed to fetch {season}. Error: {e}")

Fetching the 2021 season...
Fetching the 2022 season...
Fetching the 2023 season...
Fetching the 2024 season...
Fetching the 2025 season...


# 3. Convert the JSON response into a clean Pandas DataFrame

In [6]:
# 3. Combine all the yearly chunks into one massive Master Table
if all_seasons_data:
    df_league_master = pd.concat(all_seasons_data, ignore_index=True)
    
    print(f"\nSUCCESS! Pulled {df_league_master.shape[0]} total games across 5 years.")
    print(df_league_master[['TEAM_NAME', 'GAME_DATE', 'MATCHUP', 'PTS']].head())


SUCCESS! Pulled 2348 total games across 5 years.
            TEAM_NAME   GAME_DATE      MATCHUP  PTS
0    New York Liberty  2021-05-14  NYL vs. IND   90
1     Connecticut Sun  2021-05-14    CON @ ATL   78
2       Indiana Fever  2021-05-14    IND @ NYL   87
3     Phoenix Mercury  2021-05-14    PHO @ MIN   77
4  Los Angeles Sparks  2021-05-14  LAS vs. DAL   71


In [7]:
from sqlalchemy import create_engine

In [8]:
print("\nBuilding the SQL Server Database...")


Building the SQL Server Database...


# 4. Define your SQL Server details

In [9]:
server_name = 'Your_Server_Name' # This depends on your setup!
database_name = 'LibertyAnalytics'

# 5. Create the connection string

In [10]:
connection_string = f"mssql+pyodbc://@{server_name}/{database_name}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(connection_string)

# 6. Load the data directly into SQL Server!

In [11]:
# Dumping the massive 5-year table into SQL Server!
df_league_master.to_sql('game_logs', engine, if_exists='replace', index=False)
print("5-Year Macro Data successfully loaded into SQL Server table: 'game_logs'")

5-Year Macro Data successfully loaded into SQL Server table: 'game_logs'


# 7. Normalize Team Data

In [12]:
from sqlalchemy import text

print("Starting Database Normalization...")

# We use a 'with' block to safely open and close the database connection
with engine.connect() as conn:
    
    # 1. Build Dim_Teams (Extracting unique teams)
    print("Building Dim_Teams...")
    conn.execute(text("DROP TABLE IF EXISTS Dim_Teams;"))
    conn.execute(text("""
        SELECT DISTINCT TEAM_ID, TEAM_NAME, TEAM_ABBREVIATION
        INTO Dim_Teams
        FROM game_logs
        WHERE TEAM_ID IS NOT NULL;
    """))
    
    # 2. Build Dim_Dates (Extracting unique dates)
    print("Building Dim_Dates...")
    conn.execute(text("DROP TABLE IF EXISTS Dim_Dates;"))
    conn.execute(text("""
        SELECT DISTINCT 
            GAME_DATE,
            SUBSTRING(GAME_DATE, 1, 4) AS Game_Year,
            SUBSTRING(GAME_DATE, 6, 2) AS Game_Month
        INTO Dim_Dates
        FROM game_logs
        WHERE GAME_DATE IS NOT NULL;
    """))
    
    # 3. Build Fact_Games (The core stats)
    print("Building Fact_Games...")
    conn.execute(text("DROP TABLE IF EXISTS Fact_Games;"))
    conn.execute(text("""
        SELECT DISTINCT 
            GAME_ID, 
            TEAM_ID, 
            GAME_DATE, 
            MATCHUP, 
            WL AS Win_Loss, 
            PTS AS Points
        INTO Fact_Games
        FROM game_logs;
    """))
    
    # Save all the changes to the database
    conn.commit()

print("\nSUCCESS! Your Star Schema is complete.")

Starting Database Normalization...
Building Dim_Teams...
Building Dim_Dates...
Building Fact_Games...

SUCCESS! Your Star Schema is complete.


# 8. Fetch Last Five Years of Player Data from the API 

In [13]:
import time
from nba_api.stats.endpoints import leaguegamelog
import pandas as pd

seasons = ['2021', '2022', '2023', '2024', '2025']
wnba_league_id = '10'
all_players_data = []

print("Connecting to API to fetch 5 years of WNBA Player Box Scores...")

for season in seasons:
    print(f"Fetching Player data for the {season} season...")
    try:
        # Notice the 'P' parameter here!
        gamelog = leaguegamelog.LeagueGameLog(
            season=season, 
            league_id=wnba_league_id,
            player_or_team_abbreviation='P', 
            timeout=60 
        )
        
        df_season = gamelog.get_data_frames()[0]
        all_players_data.append(df_season)
        
        # Pause to respect the API limits
        time.sleep(3)
        
    except Exception as e:
        print(f"Failed to fetch {season}. Error: {e}")

# Combine and load the data
if all_players_data:
    df_players_master = pd.concat(all_players_data, ignore_index=True)
    print(f"\nSUCCESS! Pulled {df_players_master.shape[0]} total player box scores.")
    
    # Load this raw data into a new staging table in SQL Server
    print("Loading raw player data into SQL Server staging table...")
    df_players_master.to_sql('player_game_logs', engine, if_exists='replace', index=False)
    print("Player Staging Data successfully loaded into 'player_game_logs'")

Connecting to API to fetch 5 years of WNBA Player Box Scores...
Fetching Player data for the 2021 season...
Fetching Player data for the 2022 season...
Fetching Player data for the 2023 season...
Fetching Player data for the 2024 season...
Fetching Player data for the 2025 season...

SUCCESS! Pulled 22127 total player box scores.
Loading raw player data into SQL Server staging table...
Player Staging Data successfully loaded into 'player_game_logs'


# 9. Normalize the Player Data

In [14]:
from sqlalchemy import text

print("Starting Player Data Normalization...")

with engine.connect() as conn:
    
    # 1. Build Dim_Players (Your Master Roster)
    print("Building Dim_Players...")
    conn.execute(text("DROP TABLE IF EXISTS Dim_Players;"))
    conn.execute(text("""
        SELECT DISTINCT 
            PLAYER_ID, 
            PLAYER_NAME, 
            TEAM_ID
        INTO Dim_Players
        FROM player_game_logs
        WHERE PLAYER_ID IS NOT NULL;
    """))
    
    # 2. Build Fact_Player_Stats (The core box scores)
    print("Building Fact_Player_Stats...")
    conn.execute(text("DROP TABLE IF EXISTS Fact_Player_Stats;"))
    conn.execute(text("""
        SELECT 
            GAME_ID, 
            PLAYER_ID, 
            TEAM_ID, 
            MIN AS Minutes_Played,
            FGM AS Field_Goals_Made,
            FGA AS Field_Goals_Attempted,
            FG3M AS Three_Pointers_Made,
            FTM AS Free_Throws_Made,
            REB AS Rebounds,
            AST AS Assists,
            STL AS Steals,
            BLK AS Blocks,
            TOV AS Turnovers,
            PTS AS Points
        INTO Fact_Player_Stats
        FROM player_game_logs;
    """))
    
    conn.commit()

print("\nSUCCESS! Dim_Players and Fact_Player_Stats have been added to your Star Schema.")

Starting Player Data Normalization...
Building Dim_Players...
Building Fact_Player_Stats...

SUCCESS! Dim_Players and Fact_Player_Stats have been added to your Star Schema.


# 10. Run SQL Queries on the Database

## Which NY Liberty players average the most points during Away games in the month of August?

In [15]:
import pandas as pd

print("Executing complex Star Schema query...")

# The ultimate test of your new database architecture
complex_query = """
    SELECT 
        p.PLAYER_NAME,
        COUNT(fps.GAME_ID) AS Games_Played,
        ROUND(AVG(CAST(fps.Points AS FLOAT)), 1) AS Avg_Points,
        ROUND(AVG(CAST(fps.Assists AS FLOAT)), 1) AS Avg_Assists,
        ROUND(AVG(CAST(fps.Rebounds AS FLOAT)), 1) AS Avg_Rebounds
    FROM Fact_Player_Stats fps
    JOIN Dim_Players p 
        ON fps.PLAYER_ID = p.PLAYER_ID
    JOIN Dim_Teams t 
        ON p.TEAM_ID = t.TEAM_ID
    JOIN Fact_Games fg 
        ON fps.GAME_ID = fg.GAME_ID
    JOIN Dim_Dates d 
        ON fg.GAME_DATE = d.GAME_DATE
    WHERE t.TEAM_ABBREVIATION = 'NYL' 
      AND fg.MATCHUP LIKE '%@%'    -- The '@' symbol in the matchup string means it's an away game!
      AND d.Game_Month = '08'      -- Filtering for the late-season month of August
    GROUP BY p.PLAYER_NAME
    HAVING COUNT(fps.GAME_ID) >= 5 -- Only include players who have played at least 5 of these games
    ORDER BY Avg_Points DESC;
"""

# Run the query and load the results into a dataframe
with engine.connect() as conn:
    df_results = pd.read_sql(complex_query, conn)

print("\n--- NY Liberty: Top Away-Game Performers in August (2021-2025) ---")
print(df_results.to_string(index=False))

Executing complex Star Schema query...

--- NY Liberty: Top Away-Game Performers in August (2021-2025) ---
            PLAYER_NAME  Games_Played  Avg_Points  Avg_Assists  Avg_Rebounds
        Breanna Stewart            31        22.1          3.3           9.2
        Sabrina Ionescu            41        17.3          5.1           5.5
          Jonquel Jones            43        14.6          2.4           9.5
         Natasha Howard            42        14.0          2.3           7.1
         Emma Meesseman            19        13.3          3.8           5.2
Betnijah Laney-Hamilton            25        12.4          3.3           3.8
      Layshia Clarendon            15        10.7          4.3           3.0
          Natasha Cloud            43        10.3          5.4           3.6
   Courtney Vandersloot            31        10.1          7.4           3.5
         Leonie Fiebich            22         8.4          2.2           3.6
          Sami Whitcomb            42         

## How much does playing at the Barclays Center actually impact the Liberty's win rate year over year?

In [16]:
# Query 1: Home vs. Away Win Percentage
query_home_court = """
    SELECT 
        d.Game_Year,
        COUNT(fg.GAME_ID) AS Total_Games,
        -- Calculate Home Win Percentage
        ROUND(
            SUM(CASE WHEN fg.MATCHUP NOT LIKE '%@%' AND fg.Win_Loss = 'W' THEN 1.0 ELSE 0.0 END) / 
            NULLIF(SUM(CASE WHEN fg.MATCHUP NOT LIKE '%@%' THEN 1.0 ELSE 0.0 END), 0) * 100
        , 1) AS Home_Win_Pct,
        -- Calculate Away Win Percentage
        ROUND(
            SUM(CASE WHEN fg.MATCHUP LIKE '%@%' AND fg.Win_Loss = 'W' THEN 1.0 ELSE 0.0 END) / 
            NULLIF(SUM(CASE WHEN fg.MATCHUP LIKE '%@%' THEN 1.0 ELSE 0.0 END), 0) * 100
        , 1) AS Away_Win_Pct
    FROM Fact_Games fg
    JOIN Dim_Teams t ON fg.TEAM_ID = t.TEAM_ID
    JOIN Dim_Dates d ON fg.GAME_DATE = d.GAME_DATE
    WHERE t.TEAM_ABBREVIATION = 'NYL'
    GROUP BY d.Game_Year
    ORDER BY d.Game_Year DESC;
"""

with engine.connect() as conn:
    print("--- NY Liberty: Home vs Away Win Percentage by Year ---")
    print(pd.read_sql(query_home_court, conn).to_string(index=False))

--- NY Liberty: Home vs Away Win Percentage by Year ---
Game_Year  Total_Games  Home_Win_Pct  Away_Win_Pct
     2025           44          77.3          45.5
     2024           40          80.0          80.0
     2023           40          75.0          85.0
     2022           36          50.0          38.9
     2021           32          43.8          31.3


## Who was the number one top-scoring player for the Liberty in each of the last 5 seasons?

In [17]:
# Query 2: Year-over-Year Top Scorers using Window Functions
query_top_scorers = """
    WITH YearlyPlayerStats AS (
        SELECT 
            p.PLAYER_NAME,
            d.Game_Year,
            SUM(CAST(fps.Points AS FLOAT)) AS Total_Season_Points,
            -- This ranks players 1, 2, 3... within each specific year based on points
            ROW_NUMBER() OVER(PARTITION BY d.Game_Year ORDER BY SUM(CAST(fps.Points AS FLOAT)) DESC) as Team_Rank
        FROM Fact_Player_Stats fps
        JOIN Dim_Players p ON fps.PLAYER_ID = p.PLAYER_ID
        JOIN Dim_Teams t ON p.TEAM_ID = t.TEAM_ID
        JOIN Fact_Games fg ON fps.GAME_ID = fg.GAME_ID
        JOIN Dim_Dates d ON fg.GAME_DATE = d.GAME_DATE
        WHERE t.TEAM_ABBREVIATION = 'NYL'
        GROUP BY p.PLAYER_NAME, d.Game_Year
    )
    -- Now we just select the players who ranked #1 in their respective years
    SELECT 
        Game_Year, 
        PLAYER_NAME AS MVP_Scorer, 
        Total_Season_Points
    FROM YearlyPlayerStats
    WHERE Team_Rank = 1
    ORDER BY Game_Year DESC;
"""

with engine.connect() as conn:
    print("\n--- NY Liberty: Top Total Scorer by Season ---")
    print(pd.read_sql(query_top_scorers, conn).to_string(index=False))


--- NY Liberty: Top Total Scorer by Season ---
Game_Year      MVP_Scorer  Total_Season_Points
     2025 Sabrina Ionescu               1386.0
     2024 Breanna Stewart               1554.0
     2023 Breanna Stewart               1838.0
     2022 Breanna Stewart               1482.0
     2021 Breanna Stewart               1138.0


## When a Liberty player acts as a heavy playmaker (recording 8 or more assists in a single game), how does that impact the team's chances of winning?

In [18]:
# Query 3: Impact of High-Assist Games on Win/Loss
query_playmakers = """
    SELECT 
        p.PLAYER_NAME,
        COUNT(fps.GAME_ID) AS Games_With_8Plus_Assists,
        SUM(CASE WHEN fg.Win_Loss = 'W' THEN 1 ELSE 0 END) AS Wins_In_Those_Games,
        SUM(CASE WHEN fg.Win_Loss = 'L' THEN 1 ELSE 0 END) AS Losses_In_Those_Games
    FROM Fact_Player_Stats fps
    JOIN Dim_Players p ON fps.PLAYER_ID = p.PLAYER_ID
    JOIN Dim_Teams t ON p.TEAM_ID = t.TEAM_ID
    JOIN Fact_Games fg ON fps.GAME_ID = fg.GAME_ID
    WHERE t.TEAM_ABBREVIATION = 'NYL' 
      AND CAST(fps.Assists AS FLOAT) >= 8  -- The "Playmaker" threshold
    GROUP BY p.PLAYER_NAME
    HAVING COUNT(fps.GAME_ID) >= 3         -- Filter out statistical noise
    ORDER BY Games_With_8Plus_Assists DESC;
"""

with engine.connect() as conn:
    print("\n--- NY Liberty: Win Record When a Player Gets 8+ Assists ---")
    print(pd.read_sql(query_playmakers, conn).to_string(index=False))


--- NY Liberty: Win Record When a Player Gets 8+ Assists ---
            PLAYER_NAME  Games_With_8Plus_Assists  Wins_In_Those_Games  Losses_In_Those_Games
   Courtney Vandersloot                       108                   54                     54
          Natasha Cloud                       106                   53                     53
        Sabrina Ionescu                        90                   45                     45
      Layshia Clarendon                        14                    7                      7
Betnijah Laney-Hamilton                        10                    5                      5
          Sami Whitcomb                        10                    5                      5
         Natasha Howard                         6                    3                      3
        Stefanie Dolson                         4                    2                      2
         Emma Meesseman                         4                    2                      